# Extract established O-type loci

Picks representative loci for the established O-types from the clustering results.

Input: `oag_locus_serotype.csv`, `O-antigen_db.transposase_clean.min3orfs_rc.gbff`
Output: `Loci_Oag_Established/combined_established_otypes.gbff`

In [1]:
import pandas as pd
from Bio import SeqIO
serotype_loci = pd.read_csv("oag_locus_serotype.csv")
serotype_loci = serotype_loci[serotype_loci.serotype != "O8"]
serotype_loci = serotype_loci[serotype_loci.serotype != "O9"]
serotype_loci = serotype_loci[serotype_loci.serotype != "O62-Gp14"]
serotype_loci.index = serotype_loci.iloc[:, 0]

In [2]:
serotypes = list(set(serotype_loci.serotype))

## Representative loci

Loci with frameshifted CDS are skipped. Each serotype gets its cluster's first locus, tagged with the O-type in the source note; serotypes split across clusters get one locus per cluster.

In [3]:
oag_genbank = {}

for record in SeqIO.parse("O-antigen_db.transposase_clean.min3orfs_rc.gbff", "genbank"):
    truncation = False
    for feature in record.features:
                if feature.type == "CDS":
                    #print(feature.qualifiers.get("note", ""))
                    if "frameshift" in feature.qualifiers.get("note", [""])[0].lower():

                        truncation = True
    if not truncation:
        oag_genbank[record.id] = record
print(serotype_loci.index)
serotype_loci = serotype_loci[serotype_loci.index.isin(oag_genbank.keys())]

Index(['NZ_CP116976.1', 'NZ_CP079772.1', 'NZ_CP086627.1', 'NZ_CP188163.1',
       'NZ_CP076706.1', 'NZ_CP195014.1', 'NZ_CP135658.1', 'NZ_CP023849.1',
       'NZ_CP074726.1', 'NZ_AP039429.1',
       ...
       'NZ_CP069865.1', 'NZ_CP159652.1', 'NZ_CP074977.1', 'NZ_CP188157.1',
       'NZ_AGTD01000001.1', 'NZ_CP024659.1', 'NZ_CM165899.1', 'NZ_CP135077.1',
       'NZ_CP092500.1', 'NZ_CP021535.1'],
      dtype='str', name='Unnamed: 0', length=3635)


In [4]:
oag_serotypes = []
for serotype in serotypes:
    subset = serotype_loci[serotype_loci.serotype == serotype]
    if len(subset) == 0:
        continue
    if len(set(subset.cluster)) == 1:
        locus = subset.index[0]
        print(locus)
        record = oag_genbank[locus]

        for feature in record.features:
            if feature.type == "source":
                feature.qualifiers["note"] = [f"O locus: OL{serotype[1:]}", f"O type: {serotype}"]
        oag_serotypes.append(record)

    elif subset.cluster.value_counts().iloc[1]/subset.cluster.value_counts().sum() > 0.2:
        clusters = list(set((subset.cluster)))
        for i, cluster in enumerate(clusters):
            subset_cluster = subset[subset.cluster == cluster]
            locus = subset_cluster.iloc[0].iloc[0]
            record = oag_genbank[locus]
            for feature in record.features:
                if feature.type == "source":
                    feature.qualifiers["note"] = [f"O locus: OL{serotype[1:]}-var{i+1}", f"O type: {serotype}"]
            oag_serotypes.append(record)

SeqIO.write(oag_serotypes, "Loci_Oag_Established/combined_established_otypes.gbff", "genbank")

NZ_AP027258.1
NZ_CP100876.1
NZ_AP027648.1
NZ_CP122629.1
NZ_CP191567.1
NZ_CP138428.1
NZ_AP042512.1
NZ_CP076243.1
NZ_AP027419.1
NZ_OY754453.1
NZ_CM136167.1
NZ_CP161809.1
NZ_CP027452.1
NZ_CP123963.1
NZ_CP169224.1
NZ_CP019259.1
NZ_CP122454.1
NZ_CP027325.1
NZ_CP054556.1
NZ_CP046006.1
NZ_CP028122.1
NZ_CP122823.1
NZ_OY754409.1
NZ_CP040269.1
NZ_AP045043.1
NZ_CP101274.1
NZ_CP135658.1
NZ_CP075665.1
NZ_CP139125.1
NZ_CP117971.1
NZ_CP119740.1
NZ_CP053211.1
NZ_AP042652.1
NZ_CP116103.1
NZ_CP076706.1
NZ_AP042515.1
NZ_CM129889.1
NZ_CP063499.1
NZ_LR134296.1
NZ_CP015240.1
NZ_CP159694.1
NZ_AP027818.1
NZ_CP151045.1
NZ_CP051158.1
NZ_CP027454.1
NZ_AP027474.1
NZ_CM125665.1
NZ_CP010134.1
NZ_CM146020.1
NZ_CP051751.1
NZ_CP041437.1
NZ_CP075465.1
NZ_CP010191.1
NZ_CP148483.1
NZ_CP100944.1
NZ_AP022245.1
NZ_CM149577.1
NZ_CP035770.1
NZ_CP123593.1
NZ_CP159726.1
NZ_CP035882.1
NZ_CP055256.1
NZ_CP176489.1
NZ_CP027445.1
NZ_CP080139.1
NZ_CP020092.1
NZ_CP104788.1
NZ_CP076709.1
NZ_CM132948.1
NZ_CM129855.1
NZ_AP043621.1
NZ_AP0

165